# KDMAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import subprocess
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.KDMAge)

class KDMAge(pyagingModel):
    """Klemera-Doubal biological age with sex-specific NHANES III parameters."""

    def __init__(self):
        super().__init__()
        for sex in ["male", "female"]:
            for name in ["q", "k", "s"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))
            self.register_buffer(f"s_ba2_{sex}", torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein."""
        return log1p_crp(self.features, x)

    def postprocess(self, x):
        """Solve the Klemera-Doubal closed form for biological age.

        Notes
        -----
        The denominator sums over every biomarker, matching ``kdm_calc``'s
        deliberate choice not to rescale the estimate for missing markers.
        """
        biomarkers, age, female = x[:, :-2], x[:, -2], x[:, -1]

        def estimate(sex):
            q, k, s, s_ba2 = (
                getattr(self, f"{name}_{sex}").to(device=x.device, dty

In [3]:
model = pya.models.KDMAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "kdmage"
model.metadata["data_type"] = "clinical biomarkers"  # Paper: blood chemistry and organ function test data
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Kwon, Dayoon, and Daniel W. Belsky. \"A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge.\" GeroScience 43.6 (2021): 2795-2808."
model.metadata["doi"] = "https://doi.org/10.1007/s11357-021-00480-5"
model.metadata["notes"] = "Klemera-Doubal biological age, trained sex-specifically on NHANES III adults aged 30-75 who were not pregnant, using the BioAge package defaults. Biomarker parameters were fit on SI-unit variants so they are natively in pyaging's unit convention, and C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male parameters."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["blood"]  # Paper: blood chemistry
model.metadata["predicts"] = ["biological age"]  # Paper: Klemera-Doubal biological age
model.metadata["training_target"] = ["chronological age"]  # Paper: chronological age
model.metadata["unit"] = ["years"]  # Paper: years
model.metadata["model_type"] = "Klemera–Doubal composite"  # Paper: Klemera-Doubal biological age
model.metadata["platform"] = ["clinical laboratory assays"]  # Paper: blood chemistry and organ function test data
model.metadata["population"] = "adults"  # Paper: age >= 30 & age <= 75 & pregnant == 0
model.metadata["journal"] = "GeroScience"
model.metadata["last_author"] = "Daniel W. Belsky"
model.metadata["n_features"] = 11
model.metadata["citations"] = 332
model.metadata["citations_date"] = "2026-08-20"

## Download clock dependencies

In [5]:
# BioAge carries both the fitting functions and the NHANES III / NHANES IV tables they run
# on, so the parameters and the parity reference below are re-derived here rather than read
# from a checked-in copy. The script needs R on PATH; it installs what it is missing into a
# notebook-local library that the Clear directory step removes.
EXTRACT_R = r"""#!/usr/bin/env Rscript
# Derive the KDM biological age parameters from dayoonkwon/BioAge, which ships
# both the fitting functions and the NHANES III / NHANES IV tables they run on.
#
# The fit is trained on SI-unit variants of the NHANES columns, so the
# parameters land natively in pyaging's unit convention. KDM is scale invariant
# in its inputs -- (x - q) * k / s^2, with q, k and s all scaling with x -- so
# fitting in SI units reproduces BioAge's published numbers exactly rather than
# approximating them.
#
# Unit notes, established empirically against the shipped NHANES data:
#   * NHANES3$fev is millilitres (median 2885); NHANES3$fev_1000 is litres
#     (median 2.885). We use fev_1000.
#   * lncrp is log1p(crp in mg/dL), NOT log(crp): exp(lncrp) - crp == 1 exactly
#     across both cohorts.
#   * albumin_gL == albumin * 10 and creat_umol == creat * 88.4017 are exact,
#     with zero deviation across every non-missing row of both cohorts.
#
# CRP naming. The fitted column stays `log_crp`, because that is what the value
# is: log1p(CRP in mg/dL). The name EMITTED is `c_reactive_protein`, because
# that is what a pyaging user supplies -- the raw measurement in mg/dL, which
# the clock log1p's itself in preprocess(). The rename is name-only: no q, k or
# s moves. The emitted reference rows carry raw `crp`, so feeding them in and
# letting the clock transform reproduces BioAge's own output.

local_library <- file.path(getwd(), "Rlib")
dir.create(local_library, showWarnings = FALSE)
.libPaths(c(local_library, .libPaths()))
for (package in c("remotes", "dplyr", "jsonlite")) {
  if (!requireNamespace(package, quietly = TRUE)) {
    install.packages(package, repos = "https://cloud.r-project.org", lib = local_library)
  }
}
if (!requireNamespace("BioAge", quietly = TRUE)) {
  remotes::install_github("dayoonkwon/BioAge@b1f9fc02f086cd4aa74185f2335ab1366082e7fe", lib = local_library, upgrade = "never")
}

suppressPackageStartupMessages({
  library(BioAge)
  library(dplyr)
  library(jsonlite)
})

# Conventional -> SI conversion factors. totchol and bun have no SI column
# shipped, so they are converted here; the rest are verified against the SI
# columns BioAge already carries.
TOTCHOL_MGDL_TO_MMOL <- 0.02586 # cholesterol, 1 / 38.67
BUN_MGDL_TO_MMOL <- 0.357 # urea nitrogen, 1 / 2.8

to_pyaging_units <- function(data) {
  data %>% mutate(
    albumin = albumin_gL,
    creatinine = creat_umol,
    log_crp = lncrp,
    c_reactive_protein = crp,
    alkaline_phosphatase = alp,
    total_cholesterol = totchol * TOTCHOL_MGDL_TO_MMOL,
    blood_urea_nitrogen = bun * BUN_MGDL_TO_MMOL,
    hemoglobin_a1c = hba1c,
    systolic_blood_pressure = sbp,
    forced_expiratory_volume = fev_1000,
    female = as.numeric(gender == 2)
  )
}

nhanes3 <- to_pyaging_units(NHANES3)
nhanes4 <- to_pyaging_units(NHANES4)

to_feature_names <- function(names) replace(names, names == "log_crp", "c_reactive_protein")

markers <- c(
  "forced_expiratory_volume", "systolic_blood_pressure", "total_cholesterol",
  "hemoglobin_a1c", "albumin", "creatinine", "log_crp",
  "alkaline_phosphatase", "blood_urea_nitrogen"
)

# Same training window as BioAge::kdm_nhanes().
train_for <- function(female_value) {
  kdm_calc(
    nhanes3 %>% filter(age >= 30, age <= 75, pregnant == 0, female == female_value),
    biomarkers = markers, fit = NULL, s_ba2 = NULL
  )
}

train <- list("0" = train_for(0), "1" = train_for(1))

params_for <- function(fitted) {
  agev <- fitted$fit$lm_age
  stopifnot(identical(agev$bm, markers))
  list(
    biomarkers = to_feature_names(agev$bm),
    q = as.numeric(agev$q),
    k = as.numeric(agev$k),
    s = as.numeric(agev$s),
    s_ba2 = as.numeric(fitted$fit$s_ba2)
  )
}

# ---- Reference predictions, for the parity check ---------------------------
# 20 fixed NHANES IV subjects: complete cases across every column this clock
# consumes, sorted by sampleID (C-locale byte order), first 20. `expected`
# comes from BioAge's own kdm_calc, never from a re-implementation.
projected <- bind_rows(lapply(c(0, 1), function(female_value) {
  kdm_calc(
    nhanes4 %>% filter(female == female_value),
    biomarkers = markers,
    fit = train[[as.character(female_value)]]$fit,
    s_ba2 = train[[as.character(female_value)]]$fit$s_ba2
  )$data %>% select(sampleID, kdm)
}))

# Subject selection runs over the fitted columns, so carrying the raw CRP column
# cannot shift cohort membership and move `expected`. It is joined back after.
selected <- nhanes4 %>%
  select(sampleID, all_of(c(markers, "age", "female"))) %>%
  filter(stats::complete.cases(.)) %>%
  arrange(sampleID) %>%
  head(20) %>%
  left_join(nhanes4 %>% select(sampleID, c_reactive_protein), by = "sampleID") %>%
  left_join(projected, by = "sampleID")

stopifnot(nrow(selected) == 20, !anyNA(selected))

# The emitted rows carry raw CRP where the fit carries log1p(CRP); the clock
# closes that gap in preprocess(). If this fails, the two have drifted apart.
stopifnot(max(abs(log1p(selected$c_reactive_protein) - selected$log_crp)) < 1e-12)

emit_features <- to_feature_names(c(markers, "age", "female"))

write_json(
  list(
    features = emit_features,
    male = params_for(train[["0"]]),
    female = params_for(train[["1"]]),
    reference = list(
      sample_ids = selected$sampleID,
      rows = selected %>% select(all_of(emit_features)),
      expected = selected$kdm
    )
  ),
  "kdmage.json",
  digits = 12, auto_unbox = TRUE, pretty = TRUE
)

cat("wrote kdmage.json\n")
"""

with open("extract_kdmage.R", "w") as handle:
    handle.write(EXTRACT_R)

subprocess.run(["Rscript", "extract_kdmage.R"], check=True)

wrote kdmage.json


CompletedProcess(args=['Rscript', 'extract_kdmage.R'], returncode=0)

In [6]:
with open("kdmage.json") as handle:
    params = json.load(handle)

params["features"]

['forced_expiratory_volume',
 'systolic_blood_pressure',
 'total_cholesterol',
 'hemoglobin_a1c',
 'albumin',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'age',
 'female']

## Load features

In [7]:
model.features = params["features"]
model.features

['forced_expiratory_volume',
 'systolic_blood_pressure',
 'total_cholesterol',
 'hemoglobin_a1c',
 'albumin',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'age',
 'female']

## Load weights into base model

In [8]:
model.base_model = torch.nn.Identity()

for sex in ["male", "female"]:
    fit = params[sex]
    order = [fit["biomarkers"].index(name) for name in model.features[:-2]]
    for key in ["q", "k", "s"]:
        setattr(model, f"{key}_{sex}", torch.tensor([fit[key][index] for index in order], dtype=torch.float64))
    setattr(model, f"s_ba2_{sex}", torch.tensor(fit["s_ba2"], dtype=torch.float64))

    # The reindex above is a no-op whenever the two orders already agree, which is exactly
    # when a mangled copy of it would go unnoticed. Check it mapped what it claims.
    for position, name in enumerate(model.features[:-2]):
        source = fit["biomarkers"].index(name)
        for key in ["q", "k", "s"]:
            assert getattr(model, f"{key}_{sex}")[position].item() == fit[key][source], (sex, key, name)

model.q_female

tensor([ 3.9277, 85.5114,  3.7846,  4.4498, 41.5707, 51.6685,  0.3033, 54.9584,
         2.2111], dtype=torch.float64)

## Load reference values

In [9]:
crp = model.features.index("c_reactive_protein")
reference = [
    (male + female) / 2
    for male, female in zip(model.q_male.tolist(), model.q_female.tolist())
]
reference[crp] = math.expm1(reference[crp])  # stored raw; preprocess applies log1p

model.reference_values = reference + [52.5, 0.0]  # age: midpoint of the 30-75 window; female: male

assert len(model.reference_values) == len(model.features)
assert math.isclose(
    math.log1p(model.reference_values[crp]),
    (model.q_male[crp].item() + model.q_female[crp].item()) / 2,
)
model.reference_values

[4.617090078192,
 93.30051647306101,
 4.3532117996715005,
 4.590652017098501,
 43.670469247465505,
 60.691406723117,
 0.2575083076346611,
 65.497718435976,
 2.909765132493,
 52.5,
 0.0]

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "klemera_doubal"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for '
             'quantification of biological age from blood chemistry and organ '
             'function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'citations': 332,
 'citations_date': '2026-08-20',
 'clock_name': 'kdmage',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'journal': 'GeroScience',
 'last_author': 'Daniel W. Belsky',
 'model_type': 'Klemera–Doubal composite',
 'n_features': 11,
 'notes': 'Klemera-Doubal biological age, trained sex-specifically on NHANES '
          'III adults aged 30-75 who were not pregnant, using the BioAge '
          'package defaults. Biomarker parameters were fit on SI-unit variants '
          "so they are natively in pyaging's unit convention, and C-reactive "
  

## Normal feature ranges

In [13]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

,feature,unit,low,high
0,forced_expiratory_volume,L,0.10,8.0
1,systolic_blood_pressure,mmHg,50.00,260.0
2,total_cholesterol,mmol/L,1.00,30.0
3,hemoglobin_a1c,%,2.00,20.0
4,albumin,g/L,10.00,70.0
5,creatinine,umol/L,10.00,3000.0
6,c_reactive_protein,mg/dL,0.01,50.0
7,alkaline_phosphatase,U/L,5.00,5000.0
8,blood_urea_nitrogen,mmol/L,0.50,60.0
9,age,years,0.00,122.5


## Basic test

In [14]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[419.3437]], dtype=torch.float64)

#### Parity with BioAge

In [15]:
reference_predictions = params["reference"]
matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference_predictions["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference_predictions["expected"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 6.863842827442568e-12


## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: extract_kdmage.R
Deleted file: kdmage.json
Deleted folder: Rlib
